# 04 — Existing Chargers Baseline

Load NAP charging point dataset. Filter to stations on interurban roads only. Calculate `total_existing_stations_baseline` for File 1. Identify current coverage gaps.

## Data Inputs
- `data/processed/chargers_clean.csv`
- `data/processed/interurban_roads.parquet`

## Data Outputs
- `data/processed/interurban_chargers.csv` — existing chargers on interurban roads
- `total_existing_stations_baseline` scalar (for File_1.csv)
- `data/interim/coverage_gaps.geojson` — uncovered road segments

In [1]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')

import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from pathlib import Path
from src.constants import POWER_PER_CHARGER_KW
from src.data_loading import load_geo_parquet_compat

PROCESSED = Path('data/processed')
INTERIM = Path('data/interim')
INTERIM.mkdir(parents=True, exist_ok=True)

# Load datasets
chargers = pd.read_csv(PROCESSED / 'chargers_clean.csv')
roads = load_geo_parquet_compat(PROCESSED / 'interurban_roads.parquet')

print(f"Total NAP chargers: {chargers.shape[0]}")
print(f"Interurban road segments: {roads.shape[0]}")
print(f"\nCharger columns: {list(chargers.columns)}")
chargers.head()

Total NAP chargers: 12072
Interurban road segments: 1295

Charger columns: ['site_id', 'version', 'name', 'last_updated', 'latitude', 'longitude', 'postcode', 'address', 'operating_hours', 'num_stations', 'n_connectors', 'max_power_kw', 'connector_types', 'province']


,site_id,version,name,last_updated,latitude,longitude,postcode,address,operating_hours,num_stations,n_connectors,max_power_kw,connector_types,province
0,2023002088,NaN,Centro Porsche Baleares CP2,2023-11-21T11:39:04.000+01:00,39.5948,2.635860,7011,generalTextLine,Lunes (00:00 - 23:59) Martes (08:30 - 17:30) D...,1,1,350.0,NaN,Illes Balears
1,2023002089,NaN,Centro Porsche Baleares CP1,2023-11-21T11:43:04.000+01:00,39.5948,2.635900,7011,generalTextLine,Domingo (00:00 - 23:59) Sabado (08:30 - 17:30)...,1,1,350.0,NaN,Illes Balears
2,2023002112,NaN,Centro Porsche Alicante estación de carga 2,2023-11-21T11:56:08.000+01:00,38.3466,-0.522950,3006,generalTextLine,Lunes (00:00 - 23:59) Martes (08:30 - 20:00) M...,1,1,350.0,NaN,Alacant/Alicante
3,2023002113,NaN,Centro Porsche Alicante estación de carga 1,2023-11-21T11:59:57.000+01:00,38.3460,-0.523464,3006,generalTextLine,Lunes (00:00 - 23:59) Martes (08:30 - 20:00) M...,1,1,350.0,NaN,Alacant/Alicante
4,2023002090,NaN,Centro Porsche Asturias CP 2,2023-11-21T16:36:28.000+01:00,43.3948,-5.816220,33420,generalTextLine,NaN,1,1,320.0,NaN,Asturias


## Filter Chargers to Interurban Roads

Buffer interurban roads by 2 km. Any charger within this buffer is considered "on" an interurban corridor.

In [2]:
# Create GeoDataFrame from chargers
chargers_gdf = gpd.GeoDataFrame(
    chargers,
    geometry=gpd.points_from_xy(chargers['longitude'], chargers['latitude']),
    crs='EPSG:4326'
).reset_index(drop=True)

# Project to metric CRS for buffering (EPSG:25830 = ETRS89 UTM Zone 30N, covers Spain)
roads_utm = roads.to_crs('EPSG:25830')
chargers_utm = chargers_gdf.to_crs('EPSG:25830').reset_index(drop=True)

# Buffer roads by 2 km
BUFFER_M = 2000
road_buffer = roads_utm.geometry.buffer(BUFFER_M).union_all()
print(f"Road buffer created: 2 km around {len(roads_utm)} interurban segments")

# Spatial filter: chargers within the road buffer
mask = chargers_utm.geometry.within(road_buffer)
interurban_chargers = chargers_gdf[mask].reset_index(drop=True).copy()
interurban_chargers_utm = chargers_utm[mask].reset_index(drop=True).copy()

print(f"\nChargers near interurban roads: {len(interurban_chargers)} / {len(chargers_gdf)}")
print(f"  ({len(interurban_chargers)/len(chargers_gdf)*100:.1f}% of all NAP chargers)")

# Power stats
if 'max_power_kw' in interurban_chargers.columns:
    print(f"\nPower distribution (kW):")
    print(interurban_chargers['max_power_kw'].describe().round(1))
    fast = interurban_chargers['max_power_kw'] >= 50
    print(f"\nFast chargers (≥50 kW): {fast.sum()} ({fast.mean()*100:.1f}%)")
    ultrafast = interurban_chargers['max_power_kw'] >= 150
    print(f"Ultra-fast chargers (≥150 kW): {ultrafast.sum()} ({ultrafast.mean()*100:.1f}%)")

Road buffer created: 2 km around 1295 interurban segments

Chargers near interurban roads: 6065 / 12072
  (50.2% of all NAP chargers)

Power distribution (kW):
count    6065.0
mean       61.2
std        69.8
min         0.1
25%        22.0
50%        50.0
75%        60.0
max      1000.0
Name: max_power_kw, dtype: float64

Fast chargers (≥50 kW): 3246 (53.5%)
Ultra-fast chargers (≥150 kW): 768 (12.7%)


## Assign Chargers to Nearest Road Segment

Spatial join each charger to the nearest interurban road segment to identify which corridor it serves.

In [3]:
# Spatial join: nearest road segment for each charger
# Note: sjoin_nearest can return duplicates on ties — deduplicate by keeping closest
interurban_chargers_utm['_idx'] = range(len(interurban_chargers_utm))
joined = gpd.sjoin_nearest(
    interurban_chargers_utm[['_idx', 'geometry']],
    roads_utm[['segment_id', 'Carretera', 'road_prefix', 'is_tent', 'geometry']],
    how='left',
    distance_col='dist_to_road_m'
)
joined = joined.sort_values('dist_to_road_m').drop_duplicates(subset='_idx', keep='first')
joined = joined.set_index('_idx').sort_index()

# Add road info back
interurban_chargers['segment_id'] = joined['segment_id'].values
interurban_chargers['nearest_road'] = joined['Carretera'].values
interurban_chargers['road_prefix'] = joined['road_prefix'].values
interurban_chargers['is_tent'] = joined['is_tent'].values
interurban_chargers['dist_to_road_m'] = joined['dist_to_road_m'].values

print("Chargers per road type:")
print(interurban_chargers['road_prefix'].value_counts())
print(f"\nChargers on TEN-T corridors: {interurban_chargers['is_tent'].sum()}")
print(f"\nMean distance to road: {interurban_chargers['dist_to_road_m'].mean():.0f} m")

Chargers per road type:
road_prefix
A     2912
N     2493
AP     660
Name: count, dtype: int64

Chargers on TEN-T corridors: 3129

Mean distance to road: 720 m


## Identify Coverage Gaps

Find road segments that are farther than the maximum spacing threshold from any existing charger. These are candidate locations for new stations.

In [4]:
from src.constants import MAX_STATION_SPACING_KM, AFIR_SPACING_KM

# For each road segment, compute distance to nearest existing charger
# Use segment centroids for simplicity
seg_centroids = roads_utm.copy()
seg_centroids['geometry'] = seg_centroids.geometry.centroid

charger_points = interurban_chargers_utm[['geometry']].copy()

if len(charger_points) > 0:
    nearest = gpd.sjoin_nearest(
        seg_centroids[['segment_id', 'is_tent', 'max_spacing_km', 'geometry']],
        charger_points,
        how='left',
        distance_col='dist_to_charger_m'
    )
    # Take minimum distance per segment (in case of multiple matches)
    nearest_dist = nearest.groupby('segment_id')['dist_to_charger_m'].min().reset_index()
    roads_with_dist = roads.merge(nearest_dist, on='segment_id', how='left')

    # Convert to km
    roads_with_dist['dist_to_charger_km'] = roads_with_dist['dist_to_charger_m'] / 1000

    # A segment is a "gap" if distance to nearest charger exceeds its spacing threshold
    roads_with_dist['is_gap'] = (
        roads_with_dist['dist_to_charger_km'] > roads_with_dist['max_spacing_km']
    )

    gaps = roads_with_dist[roads_with_dist['is_gap']].copy()
    print(f"Coverage gaps: {len(gaps)} / {len(roads)} segments exceed spacing threshold")
    print(f"  TEN-T gaps (>{AFIR_SPACING_KM} km): {gaps['is_tent'].sum()}")
    print(f"  Non-TEN-T gaps (>{MAX_STATION_SPACING_KM} km): {(~gaps['is_tent']).sum()}")
    print(f"\nGap distance stats (km):")
    print(gaps['dist_to_charger_km'].describe().round(1))
else:
    roads_with_dist = roads.copy()
    roads_with_dist['dist_to_charger_km'] = np.nan
    roads_with_dist['is_gap'] = True
    gaps = roads_with_dist.copy()
    print("No interurban chargers found — all segments are gaps")

Coverage gaps: 0 / 1295 segments exceed spacing threshold
  TEN-T gaps (>60 km): 0
  Non-TEN-T gaps (>120 km): 0

Gap distance stats (km):
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: dist_to_charger_km, dtype: float64


## Summary & Save Outputs

In [5]:
# Baseline KPI for File 1
total_existing_stations_baseline = len(interurban_chargers)

print("=" * 60)
print("EXISTING CHARGERS BASELINE — SUMMARY")
print("=" * 60)
print(f"total_existing_stations_baseline = {total_existing_stations_baseline}")
print(f"\nBy road type:")
print(interurban_chargers['road_prefix'].value_counts().to_string())
print(f"\nCoverage gaps: {len(gaps)} segments need new chargers")

# Save interurban chargers CSV
drop_cols = ['geometry']
out_chargers = PROCESSED / 'interurban_chargers_baseline.csv'
interurban_chargers.drop(columns=drop_cols, errors='ignore').to_csv(out_chargers, index=False)
print(f"\nSaved: {out_chargers} ({len(interurban_chargers)} rows)")

# Save coverage gaps GeoJSON
out_gaps = INTERIM / 'coverage_gaps.geojson'
gaps.to_file(out_gaps, driver='GeoJSON')
print(f"Saved: {out_gaps} ({len(gaps)} segments)")

# Save baseline KPI as a small CSV for easy downstream use
pd.DataFrame([{
    'total_existing_stations_baseline': total_existing_stations_baseline,
}]).to_csv(PROCESSED / 'baseline_kpi.csv', index=False)
print(f"Saved: {PROCESSED / 'baseline_kpi.csv'}")

EXISTING CHARGERS BASELINE — SUMMARY
total_existing_stations_baseline = 6065

By road type:
road_prefix
A     2912
N     2493
AP     660

Coverage gaps: 0 segments need new chargers

Saved: data/processed/interurban_chargers_baseline.csv (6065 rows)
Saved: data/interim/coverage_gaps.geojson (0 segments)
Saved: data/processed/baseline_kpi.csv
